# Environment check

This notebook checks that the workspace is ready for the class laboratories. It uses a small made-up dataset and no class material.

**How to run it:** in the menu, choose **Run**, then **Run All Cells**. It takes under a minute. The last cell prints a table: every line should read **PASS**. If one reads **FAIL**, send the instructor a screenshot of that table.

In [ ]:
# Setup: a place to record each check's result
results = []
def check(name, ok, detail=""):
    results.append((name, "PASS" if ok else "FAIL", detail))
    print(("PASS" if ok else "FAIL") + "  " + name + ("  (" + detail + ")" if detail else ""))
print("ready")

In [ ]:
# Check 1: Python version (3.11 or later)
import sys
v = sys.version_info
check("Python 3.11 or later", (v.major, v.minor) >= (3, 11), f"{v.major}.{v.minor}.{v.micro}")

In [ ]:
# Check 2: the libraries the labs use, and their versions
import importlib
tested = {"numpy": "1.26.4", "pandas": "2.1.4", "scipy": "1.11.4", "sklearn": "1.2.2",
          "statsmodels": "0.14.0", "matplotlib": "3.8.0"}
for name, want in tested.items():
    try:
        mod = importlib.import_module(name)
        have = getattr(mod, "__version__", "?")
        check(f"library {name}", True, f"version {have}" + ("" if have == want else f"; the labs were tested with {want}"))
    except Exception as e:
        check(f"library {name}", False, f"not available: {e.__class__.__name__}")

In [ ]:
# Check 3: the workspace folders, and reading a data file
from pathlib import Path
import pandas as pd
here = Path.cwd()
ROOT = next((p for p in [here, *here.parents] if (p / "data").is_dir()), None)
check("workspace folder with data/ found", ROOT is not None, str(ROOT))
df = pd.read_csv(ROOT / "data" / "sample_data.csv")
check("read data/sample_data.csv", df.shape == (200, 4), f"{df.shape[0]} rows, {df.shape[1]} columns")

In [ ]:
# Check 4: tables and random numbers (pandas, NumPy)
import numpy as np
summary = df.groupby("group")["x1"].mean().round(2)
check("group summary computed", len(summary) == 2, summary.to_dict().__repr__())
rng = np.random.default_rng(2026)
first = round(float(rng.normal()), 6)
check("random numbers reproducible", abs(first - -0.793122) < 1e-9, f"first draw {first}")

In [ ]:
# Check 5: a statistical test (SciPy) and a regression table (statsmodels)
from scipy import stats
import statsmodels.formula.api as smf
t = stats.ttest_ind(df.loc[df.group == "A", "x1"], df.loc[df.group == "B", "x1"], equal_var=False)
check("t-test runs", 0 <= t.pvalue <= 1, f"p = {t.pvalue:.4f}")
ols = smf.ols("x1 ~ x2 + C(group)", data=df).fit()
check("regression fits", len(ols.params) == 3, f"R-squared {ols.rsquared:.3f}")

In [ ]:
# Check 6: a machine learning model (scikit-learn)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
X, y = df[["x1", "x2"]], df["outcome"]
scores = cross_val_score(LogisticRegression(), X, y, cv=5)
check("model trains and scores", 0.5 < scores.mean() <= 1, f"cross-validated accuracy {scores.mean():.3f}")

In [ ]:
# Check 7: a chart (matplotlib); you should see it below this cell
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 3))
for g, part in df.groupby("group"):
    ax.scatter(part["x1"], part["x2"], s=12, label=f"group {g}")
ax.set_xlabel("x1"); ax.set_ylabel("x2"); ax.legend()
plt.show()
check("chart drawn", True, "look for the scatter plot above")

In [ ]:
# Check 8: saving and reading back a file (the labs save your results in work/)
work = ROOT / "work"
work.mkdir(exist_ok=True)
(work / "environment_check.txt").write_text("saved", encoding="utf-8")
check("save and read a file in work/", (work / "environment_check.txt").read_text(encoding="utf-8") == "saved", str(work))

In [ ]:
# Summary: every line should read PASS
summary_table = pd.DataFrame(results, columns=["check", "result", "detail"])
print(summary_table.to_string(index=False))
failed = (summary_table["result"] == "FAIL").sum()
print("\nALL CHECKS PASSED: the workspace is ready." if failed == 0 else f"\n{failed} CHECK(S) FAILED: send a screenshot of this table.")